# 4장 — 온도 조절기: 균형 루프와 목표 추구

웹 챕터(4장)와 **같은 예제, 같은 수치**를 파이썬으로 계산합니다:
격차를 메우는 균형 루프(지수 감쇠, τ 뒤 63%), 반감기, 그리고 경사하강이 같은 고리라는 것.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch4')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 온도 조절기: 데우는 속도 = 격차 ÷ τ

$$\frac{dS}{dt} = \frac{G - S}{\tau} \quad\Longrightarrow\quad G - S(t) = \big(G - S(0)\big)\, e^{-t/\tau}$$

12°C 회의실, 목표 22°C, 조정 시간 τ = 10분. 정확한 답(오른쪽 식)과, 시간을 잘게 썰어 쌓은 계산(5장 오일러법)이 같은지도 봅니다.

**그림으로 읽기**(웹 4장 1절): 인과지도 `실내 온도 →(−) 목표와의 격차 →(+) 데우는 속도 →(+) 실내 온도`를 한 바퀴 따라 읽으면
온도 ↑ → 격차 ↓ → 데우는 속도 ↓ → 온도가 덜 오릅니다. 처음 변화를 거스르니 **균형 루프(B)**(− 1개). 저량-유량도로는 실내 온도 상자에
구름(난방기의 열)에서 오는 데우는 속도 파이프 하나, 그 밸브로 목표 온도(+)·조정 시간 τ(−)·실내 온도(−)가 가는 화살표로 이어져
**데우는 속도 = (목표 − 온도) ÷ τ**가 됩니다.

In [ ]:
import math

T0, G, tau = 12, 22, 10

def exact(t, tau=tau):
    return G - (G - T0) * math.exp(-t / tau)

def stepped(t_end, dt=0.001):
    # 시간을 dt 분씩 썰어 '지금 + 데우는 속도 × dt' 를 쌓는다 (5장 오일러법 미리보기)
    S = T0
    for _ in range(round(t_end / dt)):
        S += (G - S) / tau * dt
    return S

for t in [0, 5, 10, 20, 30, 60]:
    print(f'{t:3d}분  정확 {exact(t):6.2f}°C   잘게 썰어 쌓기 {stepped(t):6.2f}°C   메운 비율 {(exact(t) - T0) / (G - T0) * 100:5.1f}%')

# 웹 실험실(BalancingLab)·본문 수치 자가검증
assert round(exact(10), 1) == 18.3 and round(exact(20), 1) == 20.6 and round(exact(30), 1) == 21.5
assert abs((1 - math.exp(-1)) - 0.632) < 0.001          # τ 뒤 63%
assert abs(stepped(10) - exact(10)) < 0.01
print('✅ 웹과 일치: τ 뒤 63%(18.3°C), 2τ 86%(20.6°C), 3τ 95%(21.5°C)')

## 2. 반감기와 '왜 63%인가'

격차가 절반이 되는 시간: $e^{-t/\tau} = 0.5 \Rightarrow t = \tau \ln 2 \approx 0.69\tau$.
그리고 격차를 τ번에 나눠 매번 $1/\tau$씩 메우면 $(1 - 1/\tau)^\tau$ 가 남는데, τ가 커질수록 $e^{-1}$에 다가갑니다.

In [ ]:
half = math.log(2)
print(f'반감기 = τ × ln 2 = {half:.4f}τ  →  τ = 10분이면 {10 * half:.1f}분')
assert round(half, 2) == 0.69

for n in [1, 2, 4, 10, 100, 10000]:
    print(f'{n:6d}번에 나눠 메우면 남는 격차 {(1 - 1 / n) ** n:.4f}')
print(f'연속(매 순간 메우기): e^-1 = {math.exp(-1):.4f}')
assert round((1 - 1 / 4) ** 4, 2) == 0.32 and round(0.9 ** 10, 2) == 0.35
print('✅ 걸음 단위(τ = 4면 68% 메움)는 연속(63.2%)보다 조금 더 메운다')

## 3. 걸음 단위 균형 루프: 재고 채우기 (계산 연습)

재고 목표 100개, 지금 40개, 매주 격차의 1/4을 주문합니다. 매주 격차 × 3/4.

In [ ]:
inv, goal = 40.0, 100
history = [inv]
for week in range(4):
    inv += (goal - inv) / 4
    history.append(inv)
print('주별 재고:', [round(v, 2) for v in history])
assert history[1] == 55 and history[2] == 66.25
print('✅ 40 → 55 → 66.25 (격차 60 → 45 → 33.75)')

## 4. 경사하강은 최저점을 쫓는 균형 루프

손실 $L(\theta) = \frac{\lambda}{2}(\theta - \theta^*)^2$ 의 기울기는 $\lambda(\theta - \theta^*)$.
경사하강 한 걸음 $\theta \leftarrow \theta - \eta \lambda (\theta - \theta^*)$ 를 정리하면

$$\theta^* - \theta_{\text{다음}} = (1 - \eta\lambda)\,\big(\theta^* - \theta_{\text{지금}}\big)$$

곧 "매 걸음 격차의 ηλ만큼 메우는" 균형 루프입니다. 곡률 λ = 2, 최저점 θ* = 3, 출발 θ = −5(격차 8).

In [ ]:
lam, target, theta0 = 2, 3, -5

def gd(eta, steps=15):
    th = theta0
    out = [th]
    for _ in range(steps):
        grad = lam * (th - target)
        th = th - eta * grad
        out.append(th)
    return out

def balancing(eta, steps=15):
    # 균형 루프로 쓴 같은 계산: 격차 × (1 − ηλ)
    return [target + (theta0 - target) * (1 - eta * lam) ** k for k in range(steps + 1)]

for eta in [0.05, 0.1, 0.25, 0.4]:
    a, b = gd(eta), balancing(eta)
    assert all(abs(x - y) < 1e-12 for x, y in zip(a, b))
    half_steps = math.log(0.5) / math.log(1 - eta * lam)
    print(f'η = {eta:4.2f}: 매 걸음 격차 × {1 - eta * lam:.2f},  반이 되는 걸음 {half_steps:4.1f},  10걸음 뒤 θ = {a[10]:.3f}')

assert abs(math.log(0.5) / math.log(0.8) - 3.106) < 0.001
assert abs(gd(0.5)[1] - target) < 1e-12        # ηλ = 1 → 한 걸음에 도착
print('✅ 경사하강 = 균형 루프 (두 계산이 완전히 같음) · ηλ = 1 이면 한 걸음에 최저점')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ts = [i * 0.5 for i in range(121)]
for tau_, c in [(5, '#f43f5e'), (10, '#e11d48'), (20, '#9f1239')]:
    ax1.plot(ts, [G - (G - T0) * math.exp(-t / tau_) for t in ts], color=c, lw=2, label=f'τ = {tau_}분')
ax1.axhline(G, ls='--', color='#2563eb', lw=1)
ax1.set_xlabel('분'); ax1.set_ylabel('°C'); ax1.set_title('온도 조절기: τ가 클수록 느긋하게'); ax1.legend(); ax1.grid(alpha=0.3)
for eta, c in [(0.05, '#a5b4fc'), (0.1, '#6366f1'), (0.25, '#3730a3')]:
    ax2.plot(range(16), gd(eta), marker='o', ms=3, color=c, label=f'η = {eta}')
ax2.axhline(target, ls='--', color='#2563eb', lw=1)
ax2.set_xlabel('걸음'); ax2.set_ylabel('θ'); ax2.set_title('경사하강: 같은 균형 루프'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 오늘의 발견

- **균형 루프**: 교정 유량 = 격차 ÷ 조정 시간 τ → 격차가 같은 비율로 줄어드는 **지수 감쇠**.
- τ 뒤 63%, 2τ 뒤 86%, 3τ 뒤 95%. **반감기** = τ ln 2 ≈ 0.69τ.
- 목표가 0인 균형 루프도 흔하다(일정 비율 이탈 → 지수 감쇠). 악순환(강화 루프)과 구별.
- **경사하강 = 최저점을 쫓는 균형 루프**: 매 걸음 격차 × (1 − ηλ). 학습률 × 곡률이 1/τ 자리.
- ηλ = 1이면 한 걸음에 도착, 1을 넘으면 목표를 지나친다(5장).

다음 장(5장 시간을 잘게 썰기)에서 컴퓨터가 이런 고리의 미래를 계산하는 법을 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**